In [ ]:
import config
import xarray as xr
from xclim.core.calendar import percentile_doy
from preprocess import detect_onsets, calculate_spatial_extent, build_samples, standardize_variable, smooth_decimate, smooth_threshold
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dask_gateway import Gateway
import ast

DATA = config.SCRATCH
LOCAL = "/file/pathway"

In [ ]:
# Resolution Smoothing and Decimation
t2m_fine = xr.open_zarr(f"{DATA}/t2m.zarr")["t2m"]
t2m_c = smooth_decimate(t2m_fine)
t2m_c = t2m_c.chunk({"time": 365, "latitude": -1, "longitude": -1})
out = t2m_c.to_dataset()
for var in out.variables:
    out[var].encoding.clear()
out.to_zarr(f"{DATA}/t2m_coarse.zarr", mode="w")

In [ ]:
# Threshold Climatology
ds = xr.open_zarr(f"{DATA}/t2m_coarse.zarr")
t2m = ds["t2m"]
t2m.attrs["units"] = "K"

base = t2m.sel(time=slice(str(config.BASE_START), str(config.BASE_END)))
thresh = percentile_doy(base, window=15, per=90).sel(percentiles=90).compute()
thresh = smooth_threshold(thresh, width=31)

thresh.drop_vars("percentiles").to_dataset(name="thresh").to_zarr(f"{DATA}/thresh.zarr", mode="w")

In [ ]:
# Year Round Temperature Exceedence
t2m = xr.open_zarr(f"{DATA}/t2m_coarse.zarr")["t2m"]
thresh = xr.open_zarr(f"{DATA}/thresh.zarr")["thresh"]
doy = t2m.time.dt.dayofyear

pieces = []
step = 3650
for i in range(0, t2m.sizes["time"], step):
    sl = slice(i, i + step)
    pieces.append((t2m.isel(time=sl) > thresh.sel(dayofyear=doy.isel(time=sl))).compute())
    print(f"{i}-{min(i+step, t2m.sizes['time'])} done")

hot = xr.concat(pieces, dim="time")

In [ ]:
# Onset Detection
land = xr.open_zarr(f"{DATA}/land.zarr")["land"]
onsets = detect_onsets(hot, land, LOCAL)
print(len(onsets), "onset dates")
onsets.head()

In [ ]:
# Heatwave Detection Plot
onsets["year"] = pd.to_datetime(onsets["start_date"]).dt.year
onsets["month"] = pd.to_datetime(onsets["start_date"]).dt.month
onsets["spatial_extent"] = onsets["lon_range"].combine(onsets["lat_range"], calculate_spatial_extent).round(0)

counts = onsets.groupby("year").size()
years = counts.index.astype(int)

plt.figure(figsize=(12,6))
plt.plot(years, counts.values, marker="o", linestyle="-", label="Heatwave Count")
plt.xticks(np.arange(years.min(), years.max()+1, 5))
plt.grid(True)
plt.xlabel("Year")
plt.ylabel("Number of Heatwaves")
plt.title("Yearly Heatwave Event Count")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Smooth and decimate remaining surface variables to 48x128
for name in ["msl", "r", "u10", "v10", "tcc", "ssr"]:
    da = xr.open_zarr(f"{DATA}/{name}.zarr")[name]
    out = smooth_decimate(da).chunk({"time": 365, "latitude": -1, "longitude": -1}).to_dataset()
    for var in out.variables:
        out[var].encoding.clear()
    out.to_zarr(f"{DATA}/{name}_coarse.zarr", mode="w")
    print(f"{name} done")

In [ ]:
# Deseasonalization
SURFACE = ["msl", "t2m", "r", "u10", "v10", "tcc", "ssr"]

for name in SURFACE:
    da = xr.open_zarr(f"{DATA}/{name}_coarse.zarr")[name]
    standardize_variable(da, name, DATA)
    print(f"{name} standardized")

In [ ]:
# Sample Extraction
SURFACE = ["msl", "t2m", "r", "u10", "v10", "tcc", "ssr"]

X, dates = build_samples(f"{LOCAL}/events.csv", DATA, SURFACE)
print(X.shape)

np.save(f"{LOCAL}/samples.npy", X)
np.save(f"{LOCAL}/sample_dates.npy", np.array(dates))

In [ ]:
X = np.load(f"{LOCAL}/samples.npy")
dates = np.load(f"{LOCAL}/sample_dates.npy")
print(X.shape)  # expect (N, 7, 11, 48, 128)